# Feature engineering — Base 1 — Bitcoin

Orquestra `features/temporal.py` com as regras versionadas em `features/regras_bases.py` (`BASE_1`).

O quadro prevê um passo de `1D` (mesmo do Random Forest). O horizonte final é de um passo, conforme `projeto.yaml`.

## Disponibilidade

- Cenário de origem: **após o fechamento de `t`**, prever `t+1`.
- `Open`, `High`, `Low`, `Volume`: estado em `t` (`na_origem`), status `em_validacao`.
- `Adj Close`: **proibida** (duplica o alvo).
- Calendário cíclico da data-alvo: conhecido antecipadamente.
- Arquivos gerados: `catalogo_features_base1.csv` e `disponibilidade_covariaveis_base1.csv`.


In [1]:
from pathlib import Path
import sys

import pandas as pd

def find_project_root(start):
    for candidate in (start, *start.parents):
        if (candidate / "projeto.yaml").is_file():
            return candidate
    raise FileNotFoundError("Execute a partir do projeto ou de uma subpasta dele.")

ROOT = find_project_root(Path.cwd().resolve())
sys.path.insert(0, str(ROOT / "src"))
from series_temporais.features.regras_bases import BASE_1, validar_proibidas
from series_temporais.features.temporal import (
    build_one_step_frame,
    choque_futuro_nao_altera_features,
    chronological_train_test,
)

BASE_DIR = ROOT / "data" / "base_01"
DATA_PATH = BASE_DIR / "prepared.csv"
CATALOGO_PATH = BASE_DIR / "catalogo_features_base1.csv"
DISPONIBILIDADE_PATH = BASE_DIR / "disponibilidade_covariaveis_base1.csv"
SPEC = BASE_1
TRAIN_RATIO = 0.7


## Quadro de um passo

Cada linha é uma origem. O alvo do modelo é a variação até o passo seguinte; ele não entra nas features.

In [2]:
df = pd.read_csv(DATA_PATH)
df = df.sort_values(SPEC.time_col, kind="stable").reset_index(drop=True)
validar_proibidas(SPEC.external_cols(), SPEC)
quadro, features = build_one_step_frame(
    df,
    time_col=SPEC.time_col,
    target_col=SPEC.target_col,
    external_cols=SPEC.external_cols(),
    expected_step=SPEC.expected_step,
    lags=SPEC.lags,
    windows=SPEC.windows,
    availability=SPEC.availability(),
)
assert len(quadro) == 2805
assert len(features) == 23
assert {"y_true", "target_delta", "target_time"}.isdisjoint(features)
assert not set(SPEC.proibidas).intersection(features)
quadro[features].head()


,level_t,Open_t,High_t,Low_t,Volume_t,target_lag_1,target_lag_2,target_lag_3,target_lag_7,target_lag_14,...,target_mean_14,target_std_14,target_mean_30,target_std_30,target_hour_sin,target_hour_cos,target_dow_sin,target_dow_cos,target_month_sin,target_month_cos
0,970.403015,920.958984,972.018982,920.958984,164582000,920.382019,919.495972,921.590027,892.687012,907.937988,...,910.654428,12.593400,913.066597,75.082109,0.0,1.0,0.974928,-0.222521,0.5,0.866025
1,989.023010,970.940979,989.114014,970.742004,150110000,970.403015,920.382019,919.495972,901.541992,886.617981,...,915.116215,19.828048,912.135864,74.187190,0.0,1.0,0.433884,-0.900969,0.5,0.866025
2,1011.799988,990.000977,1013.520020,983.221008,145820992,989.023010,970.403015,920.382019,917.585999,899.072998,...,922.430860,25.919008,911.044965,72.794882,0.0,1.0,-0.433884,-0.900969,0.5,0.866025
3,1029.910034,1011.460022,1033.869995,1008.789978,201278000,1011.799988,989.023010,970.403015,919.750000,895.026001,...,930.482788,33.741360,909.976965,71.052959,0.0,1.0,-0.974928,-0.222521,0.5,0.866025
4,1042.900024,1031.329956,1045.900024,1015.159973,155064000,1029.910034,1011.799988,989.023010,921.590027,921.789001,...,940.117362,40.767419,905.816300,59.277950,0.0,1.0,-0.781831,0.623490,0.5,0.866025


## Catálogo

O arquivo CSV fica na pasta do grupo e registra, para cada coluna de `X`, se a informação é conhecida na origem, só no passado ou antecipadamente.

In [3]:
catalogo = SPEC.catalogo(features)
disponibilidade = SPEC.tabela_disponibilidade()
assert not catalogo.grupo.eq("revisar").any()
catalogo.to_csv(CATALOGO_PATH, index=False)
disponibilidade.to_csv(DISPONIBILIDADE_PATH, index=False)
disponibilidade


,coluna,disponibilidade,atraso,feature,status,justificativa
0,Open,na_origem,0,Open_t,em_validacao,Usada após o fechamento de t para prever t+1; ...
1,High,na_origem,0,High_t,em_validacao,Conhecida só após o dia t; válida no cenário p...
2,Low,na_origem,0,Low_t,em_validacao,Conhecida só após o dia t; válida no cenário p...
3,Volume,na_origem,0,Volume_t,em_validacao,"Volume acumulado de t, disponível no fechamento."
4,Adj Close,proibida,0,,excluido,Proibida no quadro modelável.


## Corte cronológico

A última linha de treino cujo alvo cruzaria a primeira origem de teste é removida.

In [4]:
treino, teste = chronological_train_test(quadro, TRAIN_RATIO)
assert len(treino) == 1962
assert len(teste) == 842
assert treino.target_time.max() < teste.origin_time.min()
pd.DataFrame({
    "recorte": ["treino", "teste"],
    "linhas": [len(treino), len(teste)],
    "primeira_origem": [treino.origin_time.min(), teste.origin_time.min()],
    "ultima_origem": [treino.origin_time.max(), teste.origin_time.max()],
})


,recorte,linhas,primeira_origem,ultima_origem
0,treino,1962,2017-01-31,2022-06-15
1,teste,842,2022-06-17,2024-10-05


## Checagem de leakage

A segunda metade da série recebe um valor artificial. As features das origens anteriores precisam permanecer idênticas.

In [5]:
resultado = choque_futuro_nao_altera_features(
    df,
    time_col=SPEC.time_col,
    indice_inicial_do_choque=len(df) // 2,
    colunas=[SPEC.target_col, *SPEC.external_cols()],
    target_col=SPEC.target_col,
    external_cols=SPEC.external_cols(),
    expected_step=SPEC.expected_step,
    lags=SPEC.lags,
    windows=SPEC.windows,
    availability=SPEC.availability(),
)
assert resultado["iguais"] is True
assert resultado["diferenca_maxima"] == 0
resultado


{'origens_comparadas': 1388, 'diferenca_maxima': 0.0, 'iguais': True}